In [6]:
"""
Semantic Search over Blog Posts
================================
Encodes a natural-language query with the same embedding model used during
ingestion, then runs vector_full_scan (exact nearest-neighbor) to find the
closest-matching blog posts.
"""

import sqlite3
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from tabulate import tabulate

# ── Paths (same as ingestion notebook) ─────────────────────────────────
DB_PATH = (
    "/home/jain/Desktop/ws/public/generative_ai_workspace_2024_04_05/"
    "14_topic_classification_of_blog_posts_using_sqlite-vector/"
    "link_to_blog (20260703_0755).db"
)
VECTOR_EXT_PATH = "/home/jain/Desktop/cupboard/program_files/vector-linux-x86_64-1.0.0/vector.so"

TABLE_NAME = "blog_posts"
TOP_K = 10


In [7]:
# ── Connect to DB & Load sqlite-vector extension ──────────────────────
conn = sqlite3.connect(DB_PATH)
conn.enable_load_extension(True)
conn.load_extension(VECTOR_EXT_PATH)

ver = conn.execute("SELECT vector_version()").fetchone()[0]
print(f"sqlite-vector: {ver}")

# ── Auto-discover embedding columns ────────────────────────────────────
NON_EMBEDDING_COLS = {
    "id", "url", "original_html", "original_text", "title",
    "new_shortened_text", "cover_image", "img_base64",
    "labels", "key_takeaways", "blogger_url", "ml_label"
}

all_cols = [c[1] for c in conn.execute(f"PRAGMA table_info({TABLE_NAME})").fetchall()]
embedding_cols = [c for c in all_cols if c not in NON_EMBEDDING_COLS]

print(f"\nFound {len(embedding_cols)} embedding column(s):")
for col in embedding_cols:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {TABLE_NAME} WHERE {col} IS NOT NULL"
    ).fetchone()[0]

    # Infer dimension from BLOB size (float32 = 4 bytes/element)
    sample = conn.execute(
        f"SELECT length({col}) FROM {TABLE_NAME} WHERE {col} IS NOT NULL LIMIT 1"
    ).fetchone()
    dim = sample[0] // 4 if sample else 0

    # vector_init is PER-CONNECTION — must be called here even if done during ingestion
    if dim:
        conn.execute(
            f"SELECT vector_init('{TABLE_NAME}', '{col}', "
            f"'type=FLOAT32,dimension={dim},distance=COSINE')"
        )

    print(f"  • {col}  ({count} rows, {dim} dims, initialized)")


sqlite-vector: 1.0.0

Found 8 embedding column(s):
  • bge_small_en_v1_5_384  (79 rows, 384 dims, initialized)
  • bge_base_en_v1_5_768  (79 rows, 768 dims, initialized)
  • e5_small_v2_384  (79 rows, 384 dims, initialized)
  • e5_base_v2_768  (79 rows, 768 dims, initialized)
  • gte_small_384  (79 rows, 384 dims, initialized)
  • gte_base_768  (79 rows, 768 dims, initialized)
  • all_minilm_l6_v2_384  (79 rows, 384 dims, initialized)
  • all_minilm_l12_v2_384  (79 rows, 384 dims, initialized)


In [8]:
# ── Map column → HuggingFace model ────────────────────────────────────
#
# The short key embedded in the column name (e.g. "bge_small_en_v1_5_384")
# is used to look up the original HuggingFace model ID.
#
# Extend this dict when you add new models in the ingestion notebook.

SHORT_TO_HF = {
    "bge_small_en_v1_5":  "BAAI/bge-small-en-v1.5",
    "bge_base_en_v1_5":   "BAAI/bge-base-en-v1.5",
    "e5_small_v2":        "intfloat/e5-small-v2",
    "e5_base_v2":         "intfloat/e5-base-v2",
    "gte_small":          "thenlper/gte-small",
    "gte_base":           "thenlper/gte-base",
    "all_minilm_l6_v2":   "sentence-transformers/all-MiniLM-L6-v2",
    "all_minilm_l12_v2":  "sentence-transformers/all-MiniLM-L12-v2",
}


def resolve_model(column_name: str) -> str:
    """
    Given a column like 'bge_small_en_v1_5_384', return the HF model ID.
    The last underscore segment is the dimension, everything before it is the short key.
    """
    # Split on last '_' — everything before it is the short key
    parts = column_name.rsplit("_", 1)
    short_key = parts[0]
    if short_key not in SHORT_TO_HF:
        raise KeyError(
            f"Unknown short key '{short_key}' from column '{column_name}'. "
            f"Add it to SHORT_TO_HF."
        )
    return SHORT_TO_HF[short_key]


# ── Query function ─────────────────────────────────────────────────────

# Cache to avoid reloading models on every query
_model_cache: dict[str, SentenceTransformer] = {}


def search(
    query_text: str,
    column: str,
    top_k: int = TOP_K,
) -> list[dict]:
    """
    Encode `query_text` with the model matching `column`, then run
    vector_full_scan for the top_k nearest neighbors.

    Returns a list of dicts with keys: rank, distance, title, text_preview, url
    """
    hf_name = resolve_model(column)

    # Load (or reuse cached) model
    if hf_name not in _model_cache:
        print(f"  Loading model: {hf_name} …")
        _model_cache[hf_name] = SentenceTransformer(hf_name, trust_remote_code=True)

    model = _model_cache[hf_name]

    # Encode query → JSON for vector_as_f32
    query_vec = model.encode(
        query_text,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    query_json = json.dumps(query_vec.tolist())

    # Run exact nearest-neighbor search
    sql = f"""
        SELECT
            v.rowid,
            v.distance,
            b.title,
            b.original_text,
            b.url
        FROM vector_full_scan(
            '{TABLE_NAME}',
            '{column}',
            vector_as_f32(?),
            ?
        ) AS v
        JOIN {TABLE_NAME} AS b ON b.id = v.rowid
    """
    rows = conn.execute(sql, (query_json, top_k)).fetchall()

    results = []
    for rank, (rowid, distance, title, text, url) in enumerate(rows, start=1):
        preview = (text or "")[:200].replace("\n", " ")
        results.append({
            "rank": rank,
            "distance": round(distance, 6),
            "title": title or "(no title)",
            "text_preview": preview + ("…" if len(text or "") > 200 else ""),
            "url": url or "",
        })

    return results


In [9]:
# ── Demo: pick a column & run a query ──────────────────────────────────
#
# Change these two variables and re-run this cell for different searches.

DEMO_QUERY = "Your Why Matters More Than Your How"
DEMO_COLUMN = embedding_cols[0] if embedding_cols else None

if DEMO_COLUMN is None:
    print("No embedding columns found. Run the ingestion notebook first.")
else:
    print(f"Query    : \"{DEMO_QUERY}\"")
    print(f"Column   : {DEMO_COLUMN}")
    print(f"Model    : {resolve_model(DEMO_COLUMN)}")
    print("-" * 70)

    results = search(DEMO_QUERY, DEMO_COLUMN, top_k=TOP_K)

    if results:
        table = [
            [r["rank"], r["distance"], r["title"], r["text_preview"]]
            for r in results
        ]
        print(tabulate(
            table,
            headers=["#", "Distance", "Title", "Preview"],
            tablefmt="simple",
            maxcolwidths=[None, None, 40, 60],
        ))
        print(f"\nShowing {len(results)} closest matches (COSINE distance, lower = more similar).")
    else:
        print("No results returned.")


Query    : "Your Why Matters More Than Your How"
Column   : bge_small_en_v1_5_384
Model    : BAAI/bge-small-en-v1.5
----------------------------------------------------------------------
  Loading model: BAAI/bge-small-en-v1.5 …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  #    Distance  Title                                     Preview
---  ----------  ----------------------------------------  ------------------------------------------------------------
  1    0.360401  Failures Are Life's Greatest Teacher: An  Chinese proverb of the day on failure life lessons: Chinese
                 Ancient Chinese Proverb                   proverb of the day: 'Every time you fail, you...' – this
                                                           Chinese saying reveals why failure and setbacks can become
                                                           life’s greatest lessons …
  2    0.360719  Letting Go: The Superpower That Unlocks   The hidden path to wisdom and success: Zen proverb of the
                 Wisdom and Success                        day: "Knowledge is learning something every day. Wisdom is
                                                           letting go of something every day." Knowledge vs wisdom? How
                    

In [10]:
# ── Compare results across ALL models ─────────────────────────────────
#
# Run the same query against every embedding column to see how different
# models rank the same content.  Useful for model evaluation.

# COMPARE_QUERY = "how to fine-tune large language models efficiently"
COMPARE_QUERY = DEMO_QUERY

print(f"Query: \"{COMPARE_QUERY}\"\n")

for col in embedding_cols:
    results = search(COMPARE_QUERY, col, top_k=3)
    print(f"── {col} ({resolve_model(col)}) ──")
    for r in results:
        print(f"  #{r['rank']}  d={r['distance']:.4f}  {r['title'][:70]}")
    print()


Query: "Your Why Matters More Than Your How"

── bge_small_en_v1_5_384 (BAAI/bge-small-en-v1.5) ──
  #1  d=0.3604  Failures Are Life's Greatest Teacher: An Ancient Chinese Proverb
  #2  d=0.3607  Letting Go: The Superpower That Unlocks Wisdom and Success
  #3  d=0.3748  The Test Before the Lesson: Why Experience Teaches Differently

  Loading model: BAAI/bge-base-en-v1.5 …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

── bge_base_en_v1_5_768 (BAAI/bge-base-en-v1.5) ──
  #1  d=0.4365  Your Why Matters More Than Your How
  #2  d=0.4385  The Test Before the Lesson: Why Experience Teaches Differently
  #3  d=0.4541  Remember Every Drop: The Power of Self-Reliance

  Loading model: intfloat/e5-small-v2 …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

── e5_small_v2_384 (intfloat/e5-small-v2) ──
  #1  d=0.1792  Your Why Matters More Than Your How
  #2  d=0.1927  Meta’s Ad-Free Subscription Arrives Under Stricter Privacy Laws
  #3  d=0.1930  The Ancient Cure for a Restless Mind

  Loading model: intfloat/e5-base-v2 …


...

── gte_base_768 (thenlper/gte-base) ──
  #1  d=0.1877  Your Why Matters More Than Your How
  #2  d=0.2042  Remember Every Drop: The Power of Self-Reliance
  #3  d=0.2072  Cultivate Your Field: The Ancient African Wisdom on Why Hard Work Stil

  Loading model: sentence-transformers/all-MiniLM-L6-v2 …


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

── all_minilm_l6_v2_384 (sentence-transformers/all-MiniLM-L6-v2) ──
  #1  d=0.7663  Your Why Matters More Than Your How
  #2  d=0.7934  Crying Together: Anne Frank's Lesson on Healing
  #3  d=0.7987  The Quiet Wisdom of Knowing Your Child

  Loading model: sentence-transformers/all-MiniLM-L12-v2 …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

── all_minilm_l12_v2_384 (sentence-transformers/all-MiniLM-L12-v2) ──
  #1  d=0.6739  Your Why Matters More Than Your How
  #2  d=0.7336  The Test Before the Lesson: Why Experience Teaches Differently
  #3  d=0.7516  Crying Together: Anne Frank's Lesson on Healing

